### RAG Pipelines - Data Ingestion to vector DB Pipeline

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\yellu\AppData\Local\Temp\ipykernel_31560\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\yellu\OneDrive\Desktop\cloud_colab\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
## Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")


    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFS in the data directory
all_pdf_documents = process_all_pdfs("../data")



Found 4 PDF files to process

Processing: attention.pdf
 Loaded 8 pages

Processing: embeddings.pdf
 Loaded 8 pages

Processing: object_detection.pdf
 Loaded 8 pages

Processing: proposal.pdf
 Loaded 7 pages

Total documents loaded: 31


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T05:22:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T05:22:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Educational document created for RAG learning\nPage 1\n Attention and Transformer Fundamentals\n A practical educational document for learning Retrieval-Augmented Generation (RAG)\nPurpose: This document introduces the attention mechanism and the Transformer architecture in a way that is useful\nfor understanding modern language models and RAG systems.\nLearning goals\n \x7f Understand why attention was introduced and what problem it solves.\n \x7f Understand queries, keys, and values (Q, K, V).\n \x7f Understand scaled dot-product attent

In [8]:
## text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [9]:
chunks = split_documents(all_pdf_documents)
chunks

Split 31 documents into 54 chunks

Example chunk:
Content: Educational document created for RAG learning
Page 1
 Attention and Transformer Fundamentals
 A practical educational document for learning Retrieval-Augmented Generation (RAG)
Purpose: This document ...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T05:22:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T05:22:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T05:22:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T05:22:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Educational document created for RAG learning\nPage 1\n Attention and Transformer Fundamentals\n A practical educational document for learning Retrieval-Augmented Generation (RAG)\nPurpose: This document introduces the attention mechanism and the Transformer architecture in a way that is useful\nfor understanding modern language models and RAG systems.\nLearning goals\n \x7f Understand why attention was introduced and what problem it solves.\n \x7f Understand queries, keys, and values (Q, K, V).\n \x7f Understand scaled dot-product attent

### Embedding And VectorstoreDB

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded sucessfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text string to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager




Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2295.77it/s]


Model loaded sucessfully. Embedding dimension: 384


### Vectore Store

In [12]:
class vectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """

        Intialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """Initialize chromaDB client and collection"""

        try:
            # create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collections: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorStore = vectorStore()
vectorStore

Vector store initialized. Collection: pdf_documents
Existing documents in collections: 162


In [13]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T05:22:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T05:22:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Educational document created for RAG learning\nPage 1\n Attention and Transformer Fundamentals\n A practical educational document for learning Retrieval-Augmented Generation (RAG)\nPurpose: This document introduces the attention mechanism and the Transformer architecture in a way that is useful\nfor understanding modern language models and RAG systems.\nLearning goals\n \x7f Understand why attention was introduced and what problem it solves.\n \x7f Understand queries, keys, and values (Q, K, V).\n \x7f Understand scaled dot-product attent

In [14]:
### Convert the text to embeddings 
texts = [doc.page_content for doc in chunks]

## generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vectorStore.add_documents(chunks, embeddings)

Generating embeddings for 54 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.00s/it]

Generated embeddings with shape: (54, 384)
Adding 54 documents to vector store...
Successfully added 54 documents to vector store
Total documents in collection: 216


### Retriever Pipeline From vectorStore

In [15]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: vectorStore, embedding_manager: EmbeddingManager):
        """

        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """

        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results=top_k
            )

            # Process results

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert disatnce to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id'               : doc_id,
                            'content'          : document,
                            'metadata'         : metadata,
                            'similarity_score' : similarity_score,
                            'distance'         : distance,
                            'rank'             : i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorStore, embedding_manager)



In [16]:
rag_retriever

In [17]:
rag_retriever.retrieve("What is attention is all you need")

retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [18]:
vectorStore.collection.count()

216

In [19]:
results = vectorStore.collection.query(
    query_embeddings=[
        embedding_manager.generate_embeddings(
            ["What is attention is all you need"]
        )[0].tolist()
    ],
    n_results=5
)

results

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 88.01it/s]

Generated embeddings with shape: (1, 384)


{'ids': [['doc_88a82bee_4',
   'doc_e5195669_4',
   'doc_cf0f019d_4',
   'doc_a540b37d_4',
   'doc_489fe179_15']],
 'embeddings': None,
 'documents': [["Educational document created for RAG learning\nPage 3\n2. Queries, Keys, and Values\nSelf-attention is commonly described using three learned projections: Query (Q), Key (K), and Value (V). They are\nproduced from the token representations.\nComponent\nIntuition\nRole\nQuery\nWhat am I looking for?\nRepresents the current token's information need.\nKey\nWhat information do I contain?\nUsed to compare with a query.\nValue\nWhat information should I pass on?\nContent aggregated after attention weights are calculated.\nAnalogy: library search\nImagine a library visitor has a question. The question is the Query. Each book has searchable metadata that acts like a\nKey. The actual useful material inside the selected books acts like the Values. Attention calculates which keys best\nmatch the query and then combines the corresponding values.\n

In [20]:
vectorStore.collection.metadata

{'description': 'PDF document embeddings for RAG'}

### Verify the distance metric

In [21]:
vectorStore.collection.configuration

{'hnsw': {'space': 'l2',
  'ef_construction': 100,
  'ef_search': 100,
  'max_neighbors': 16,
  'resize_factor': 1.2,
  'sync_threshold': 1000},
 'spann': None,
 'embedding_function': <chromadb.api.types.DefaultEmbeddingFunction at 0x270cc5f1cd0>}

In [22]:
vectorStore.collection.get(
    include=["documents", "metadatas", "embeddings"]
)

{'ids': ['doc_58ae7dc4_0',
  'doc_cf51d30d_1',
  'doc_59aa044b_2',
  'doc_3020d050_3',
  'doc_88a82bee_4',
  'doc_f841c0e8_5',
  'doc_aa76a54a_6',
  'doc_75ff7b46_7',
  'doc_32f09c95_8',
  'doc_13158b2e_9',
  'doc_a5fe4b61_10',
  'doc_5c858d24_11',
  'doc_667061ef_12',
  'doc_844957c3_13',
  'doc_bdf6b184_14',
  'doc_489fe179_15',
  'doc_90803af2_16',
  'doc_c439d8b3_17',
  'doc_5de08593_18',
  'doc_a1b06135_19',
  'doc_276530d9_20',
  'doc_87cb0ebf_21',
  'doc_dd1f7e54_22',
  'doc_6e750356_23',
  'doc_40ca2c08_24',
  'doc_c7a94873_25',
  'doc_a2fd7ba0_26',
  'doc_07691a32_27',
  'doc_f7a19343_28',
  'doc_5fbd9d5b_29',
  'doc_d46bee20_30',
  'doc_197d8832_31',
  'doc_3b8aebc6_32',
  'doc_ac927180_33',
  'doc_1484ce67_34',
  'doc_4b3d085f_35',
  'doc_2f8270c7_36',
  'doc_1ce01abe_37',
  'doc_e201d58c_38',
  'doc_4a9fdb4c_39',
  'doc_9259e81c_40',
  'doc_0994cd91_41',
  'doc_ca3b276f_42',
  'doc_71d38ef1_43',
  'doc_cc15e3e9_44',
  'doc_05b231cb_45',
  'doc_f96d757d_46',
  'doc_3ec80b94_

### Create a new cosine collection

In [23]:
cosine_collection = vectorStore.client.get_or_create_collection(
    name="pdf_documents_cosine",
    metadata={
        "description": "PDF document embeddings for RAG - cosine distance",
        "hnsw:space": "cosine"
    }
)

print("Collection created:", cosine_collection.name)
print("Documents currently in collection:", cosine_collection.count())

Collection created: pdf_documents_cosine
Documents currently in collection: 162


In [24]:
cosine_collection.configuration

{'hnsw': {'space': 'cosine',
  'ef_construction': 100,
  'ef_search': 100,
  'max_neighbors': 16,
  'resize_factor': 1.2,
  'sync_threshold': 1000},
 'spann': None,
 'embedding_function': <chromadb.api.types.DefaultEmbeddingFunction at 0x270ccd69250>}

### Get the existing 108 documents

In [25]:
old_data = vectorStore.collection.get(
    include=["documents", "metadatas", "embeddings"]
)

print("IDs:", len(old_data["ids"]))
print("Documents:", len(old_data["documents"]))
print("Metadata:", len(old_data["metadatas"]))
print("Embeddings:", len(old_data["embeddings"]))

IDs: 216
Documents: 216
Metadata: 216
Embeddings: 216


### Copy the 108 records into the cosine collection

In [26]:
cosine_collection.add(
    ids=old_data["ids"],
    documents=old_data["documents"],
    metadatas=old_data["metadatas"],
    embeddings=old_data["embeddings"]
)

print("Documents in cosine collection:", cosine_collection.count())

Documents in cosine collection: 216


### Verify the new collection one more time

In [27]:
results_cosine = cosine_collection.query(
    query_embeddings=[
        embedding_manager.generate_embeddings(
            ["What is attention is all you need"]
        )[0].tolist()
    ],
    n_results=5
)

print("IDs:", results_cosine["ids"][0])
print("Distances:", results_cosine["distances"][0])

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.96it/s]

Generated embeddings with shape: (1, 384)
IDs: ['doc_88a82bee_4', 'doc_e5195669_4', 'doc_cf0f019d_4', 'doc_a540b37d_4', 'doc_29a32d5e_15']
Distances: [0.5681838989257812, 0.5681838989257812, 0.5681838989257812, 0.5681838989257812, 0.5855045914649963]


### Connect RAGRetriever to the cosine collection

In [28]:
vectorStore.collection = cosine_collection

print("Current collection:", vectorStore.collection.name)
print("Documents:", vectorStore.collection.count())
print("Metric:", vectorStore.collection.configuration["hnsw"]["space"])

Current collection: pdf_documents_cosine
Documents: 216
Metric: cosine


### Re-create the retriever

In [29]:
rag_retriever = RAGRetriever(
    vector_store=vectorStore,
    embedding_manager=embedding_manager
)

print("Retriever created successfully")

Retriever created successfully


### Test RAGRetriever

In [30]:
retrieved_docs = rag_retriever.retrieve(
    "What is attention is all you need"
)

print("\nNumber of retrieved documents:", len(retrieved_docs))

for doc in retrieved_docs:
    print("\nRank:", doc["rank"])
    print("Similarity:", doc["similarity_score"])
    print("Distance:", doc["distance"])
    print("Content:", doc["content"][:200])

retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)

Number of retrieved documents: 5

Rank: 1
Similarity: 0.43181610107421875
Distance: 0.5681838989257812
Content: Educational document created for RAG learning
Page 3
2. Queries, Keys, and Values
Self-attention is commonly described using three learned projections: Query (Q), Key (K), and Value (V). They are
prod

Rank: 2
Similarity: 0.43181610107421875
Distance: 0.5681838989257812
Content: Educational document created for RAG learning
Page 3
2. Queries, Keys, and Values
Self-attention is commonly described using three learned projections: Query (Q), Key (K), and Value (V). They are
prod

Rank: 3
Similarity: 0.43181610107421875
Distance: 0.5681838989257812
Content: Educational document created for RAG learning
Page 3
2. Queries, Keys, and Values
Self-attention is commonly described using three learned projections: Query (Q), Key (K), and Value (V). They are
prod

Rank: 4
Similarity: 0.43181610107421875
Dis

### Integration Vectordb Context pipeline With LLM output

In [35]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

## Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## Generate the answer Using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}
        
        Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

### Enhanced RAG Pipeline Fetaures

In [36]:
# ---- Enhanced RAG Pipeline Features ----
def rag_advanced(query, retriever, llm, top_k=5, score_threshold=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    -  Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)
    if not results:
        return {'answer' : 'No relevant context found.', 'sources': [], 'confidence' : 0.0, 'context' : ''}

    ## prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source'  : doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page'    : doc['metadata'].get('page', 'unknown'),
        'score'   : doc['similarity_score'],
        'preview' : doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer'     : response.content,
        'sources'    : sources,
        'confidence' : confidence
    }
    if return_context:
        output['context'] = context
    return output

## example usage
result = rag_advanced("Scaled Dot-Product Attention?", rag_retriever, llm, top_k=3, score_threshold=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

retrieving documents for query: 'Scaled Dot-Product Attention?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **Scaled Dot‑Product Attention**

\[
\text{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^{\top}}{\sqrt{d}}\right)V
\]

1. **Score** – Compute dot‑products between each query and all keys.  
2. **Scale** – Divide by \(\sqrt{d}\) (the key dimension) to keep the logits at a stable magnitude.  
3. **Normalize** – Apply softmax to obtain weights that sum to ~1, then use those weights to combine the value vectors.  

The result is a weighted sum of the value vectors, where the weights reflect the similarity between queries and keys.
Sources: [{'source': 'attention.pdf', 'page': 3, 'score': 0.6554749608039856, 'preview': 'Educational document created for RAG learning\nPage 4\n3. Scaled Dot-Product Attention\n The standard attention calculation is often written as:\nAttention(Q, K, V) = softmax(QK■ / √d■)V\nThe process has three conceptual stages.\n \x7f Score: Compute similarities between queries and keys using a dot product....'}, {'source': 'attention.pdf', 'page': 3, 'sc